# Reproducing *The Prompt Proposes, the Code Decides*

This notebook regenerates the paper's tables from the saved per-task and per-call results in
[github.com/RobertoVetere/nervo-gemma-paper](https://github.com/RobertoVetere/nervo-gemma-paper). It needs no GPU,
no model and no API key (internet on, to clone the repository). Each section prints the numbers next to where they
appear in the paper. Re-running the experiments themselves is described in `docs/REPRODUCE.md`.

In [ ]:
import os, subprocess
if not os.path.isdir('nervo-gemma-paper'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RobertoVetere/nervo-gemma-paper'], check=True)
os.chdir('nervo-gemma-paper')
def run(cmd):
    print('$', cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(p.stdout, p.stderr)

## 4.2 Five small models, three interfaces, short and long context

80 blocks of real code × 2 samples × 3 interfaces on our vLLM 0.19.1 server (each family's tool parser), original and
4-bit weights. *Structural failures* (call never runs or loses a field) is the paper's table; then the paired
block-level McNemar tests. Rows `31B (API; asked for e4b)` were requested as E4B from an API that served a 31B (paper, 4.6).
Pre-registered hypotheses and outcomes: `docs/HYPOTHESES_12NOV.md`.

In [ ]:
run('python3 -I lab/oneshot/summary_format.py results/call_format/*.jsonl | sed -n "/Structural/,\$p"')

In [ ]:
run('cat docs/HYPOTHESES_12NOV.md')

## 4.3 The same agent with two interfaces (E4B 4-bit, 40 tasks)

In [ ]:
run('python3 -I local/count_ab.py results/modal_e4b_4bit/ab_calls.jsonl results/modal_e4b_4bit/ab')

## 4.4 Reasoning × agent structure: 31B (API), three replicas; E4B 4-bit, two replicas

In [ ]:
run('python3 -I local/replica_table.py results/local_deepinfra/l40 results/local_deepinfra/l40_r2 results/local_deepinfra/l40_r3')
run('python3 -I local/replica_table.py results/local_deepinfra/l40_r2 results/local_deepinfra/l40_r3 | tail -4')
run('python3 -I local/replica_table.py results/modal_e4b_4bit/q1 results/modal_e4b_4bit/q2')

## 4.5 Single calls: E4B bf16 vs 4-bit on one server, 31B (API)

In [ ]:
run('python3 -I lab/oneshot/summary_quantization.py')

## 4.1 Tool-call failures in agent runs (one row per evaluation run, with its run id)

In [ ]:
run('cat results/format_agents/SUMMARY.md')